# PD14 network write example

Parameterizes and executes a `PD14NetworkWriteTask`, which writes a point-neuron cortical
microcircuit (Potjans & Diesmann 2014) to SONATA files.

This example reproduces the **default configuration** of `microcircuit.sonata_network.SonataNetwork`:
the eight populations `L23E, L23I, L4E, L4I, L5E, L5I, L6E, L6I`, their default neuron counts,
external indegrees, firing rates, initial membrane potentials, layer geometry, the default
connection-probability matrix, and the default thalamic input. The parameters are read from the
`microcircuit` defaults (`default_net_dict` / `default_stim_dict`) and translated into
`PointNeuronPopulation` and `PathwayConnectionProbability` blocks, so the resulting SONATA circuit
matches the model's default network.

In [ ]:
from pathlib import Path
import obi_auth

from microcircuit.network_params import default_net_dict
from microcircuit.stimulus_params import default_stim_dict

from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.info import Info
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.blocks.point_neuron_populations.pathway import (
    PathwayConnectionProbability,
)
from obi_one.scientific.blocks.point_neuron_populations.population import (
    PointNeuronPopulation,
)
from obi_one.scientific.tasks.pd14_network_write import PD14NetworkWriteScanConfig

from entitysdk import Client
from obi_notebook.get_projects import get_projects

In [ ]:
output_root = Path("../../../../../../obi-output/pd14_network_write")

### Read the default microcircuit parameters

All per-population and global values below come from the `microcircuit` defaults, so the example
stays in sync with the model rather than hard-coding numbers.

In [ ]:
population_names = list(default_net_dict["populations"])
num_neurons = default_net_dict["full_num_neurons"]
mean_rates = default_net_dict["full_mean_rates"]
k_ext = default_net_dict["K_ext"]
conn_probs = default_net_dict["conn_probs"]  # conn_probs[target][source]
psp_matrix = default_net_dict["PSP_matrix_mean"]  # psp_matrix[target][source] (mV)
weight_rel_std = default_net_dict["weight_rel_std"]
psp_ext = default_net_dict["PSP_exc_mean"]
v0_mean = default_net_dict["neuron_params"]["V0_mean"]["optimized"]
v0_std = default_net_dict["neuron_params"]["V0_std"]["optimized"]
layer_geometry = default_net_dict["layer_geometry"]
conn_probs_th = default_stim_dict["conn_probs_th"]  # thalamus -> population
num_th_neurons = default_stim_dict["num_th_neurons"]

# In the default model, populations alternate excitatory / inhibitory (even index excitatory).
synapse_classes = ["EXC" if i % 2 == 0 else "INH" for i in range(len(population_names))]
population_names

### Build one `PointNeuronPopulation` block per population

Each block carries everything the exporter sets per population: name, neuron count, synapse class,
external indegree, mean firing rate, spatial extent, initial membrane potential distribution, and
the thalamo-cortical connection probability.

In [ ]:
populations = {}
for i, name in enumerate(population_names):
    geometry = layer_geometry[name]
    populations[name] = PointNeuronPopulation(
        population_name=name,
        num_neurons=int(num_neurons[i]),
        synapse_class=synapse_classes[i],
        k_ext=int(k_ext[i]),
        mean_rate=float(mean_rates[i]),
        x_min=float(geometry["x"][0]),
        x_max=float(geometry["x"][1]),
        y_min=float(geometry["y"][0]),
        y_max=float(geometry["y"][1]),
        z_min=float(geometry["z"][0]),
        z_max=float(geometry["z"][1]),
        v0_mean=float(v0_mean[i]),
        v0_std=float(v0_std[i]),
        thalamic_connection_probability=float(conn_probs_th[i]),
    )
populations

### Assemble the scan config and add the populations

The global parameters in `Initialize` are left at their defaults (which match the model defaults),
except for enabling the thalamus via `num_th_neurons`. Populations are added first so that each
block obtains a reference (`population.ref`) that the pathway blocks can point to.

In [ ]:
scan_config = PD14NetworkWriteScanConfig.empty_config()
scan_config.set(
    PD14NetworkWriteScanConfig.Initialize(
        num_th_neurons=int(num_th_neurons),
        psp_ext=float(psp_ext),
        weight_th_rel_std=float(weight_rel_std),
        n_scaling=0.2,
        k_scaling=0.2,
    ),
    name="initialize",
)
scan_config.set(
    Info(
        campaign_name="PD14Network - second test",
        campaign_description="First attempts with scaling factors 0.2"
    ),
    name="info"
)

for name, population in populations.items():
    scan_config.add(population, name=name)

### Add the connection-probability matrix as `PathwayConnectionProbability` blocks

The default `conn_probs` matrix is indexed `conn_probs[target][source]`. We add one pathway block
for every non-zero entry; zero entries are left out and have no connections.
Each pathway also takes its mean PSP (the absolute value of `PSP_matrix_mean`) and the relative weight standard deviation from the defaults.

In [ ]:
for i, target_name in enumerate(population_names):
    for j, source_name in enumerate(population_names):
        probability = float(conn_probs[i][j])
        if probability == 0.0:
            continue
        scan_config.add(
            PathwayConnectionProbability(
                source=populations[source_name].ref,
                target=populations[target_name].ref,
                connection_probability=probability,
                psp_mean=abs(float(psp_matrix[i][j])),
                weight_rel_std=float(weight_rel_std),
            ),
            name=f"{source_name}__{target_name}",
        )

scan_config = scan_config.validated_config()
scan_config

### Platform authentication - generate Client

In [ ]:
token = obi_auth.get_token(environment="staging", auth_mode="daf")
project_context = get_projects(token, env="staging")

### Generate the single-coordinate scan and execute the task

In [ ]:
entity_client = Client(token_manager=token, project_context=project_context, environment="staging")

grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=str(output_root),
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()
run_tasks_for_generated_scan(
    grid_scan,
    db_client=entity_client
)

### Inspect the written SONATA circuit

In [ ]:
import libsonata

instance = grid_scan.single_configs[0]
circuit_root = Path(instance.coordinate_output_root)
print("Written to:", circuit_root)
print("Files:", sorted(p.name for p in circuit_root.iterdir()))

circuit = libsonata.CircuitConfig.from_file(str(circuit_root / "circuit_config.json"))
print("\nNode populations:", sorted(circuit.node_populations))
print("Edge populations:", sorted(circuit.edge_populations))

nodes = circuit.node_population("microcircuit")
print("\nTotal cortical neurons:", nodes.size)
edges = circuit.edge_population("microcircuit__microcircuit__chemical")
print("Total recurrent connections:", edges.size)